# Week 11 · Day 2 — Data Engineering & the Evaluation Benchmark

**Goal today:** turn your raw, messy dataset into clean, *leakage-safe* training data, and start the 100-question test we'll grade the model on later. **No GPU needed today** — leave the accelerator off to save your Kaggle quota.

This notebook is written to *teach*. For each step it explains **What** we do, **Why** it matters, **What could go wrong**, and **How we check it worked**.

## Where we are
- **Day 1 (done):** set up the project and inspected the data. Your original 638 rows were only **88 unique**, so we added two public datasets (electrical-engineering + physics) → about **3,200 unique** examples.
- **Day 2 (today):** clean → remove duplicates → split into train/validation *without leakage* → save → start the 100-question multiple-choice benchmark.

## How to run this on Kaggle
1. Click **+ Add Input** (right sidebar) and attach the dataset you uploaded. Kaggle mounts it read-only under `/kaggle/input/<your-dataset-name>/`.
2. Run the cells top to bottom.
3. When done, click **Save Version** (top-right) so the files written to `/kaggle/working/` are kept.

## The 4 ideas you'll learn today
- **Data leakage** — if a question (or a near-copy) appears in *both* training and test/validation, the model has effectively seen the answer key. Scores look great but are **fake**. Avoiding this is today's #1 job.
- **Deduplication** — the same question can appear many times; we keep **one copy** of each so the model doesn't just memorize repeats.
- **Train / Validation split** — *train* = what the model studies; *validation* = a held-out set we watch during training to catch **overfitting** (memorizing instead of learning).
- **A held-out benchmark (the 100 MCQs)** — a separate, hand-written exam the model never saw, used at the very end to judge improvement fairly.


In [ ]:
# ---------- Setup: imports + small helper functions (each explained inline) ----------
import json, os, re, random, glob
from collections import Counter, defaultdict

SEED = 3407              # fixing the "random seed" makes every run reproducible
random.seed(SEED)

def read_jsonl(path):
    """Read a file with one JSON object per line (skips blank/broken lines)."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows

def norm(text):
    """Normalize text so tiny differences don't count as 'different' questions:
    lowercase, drop punctuation, collapse spaces."""
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def word_set(text):
    return set(norm(text).split())

def jaccard(a, b):
    """Similarity of two word-sets: shared words / total words (0=nothing, 1=identical)."""
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

SYSTEM_PROMPT = ("You are an expert RF, DSP, and wireless communications engineer. "
                 "Answer precisely and show the key steps. When code is required, "
                 "return correct, runnable Python.")

def build_answer(r):
    """Turn one dataset row into the assistant's answer text (handles all task types)."""
    t = r.get("task_type", "")
    if t == "code":
        out = (r.get("output") or "").strip()
        return f"```python\n{out}\n```" if out else ""
    if t == "numeric":
        parts = []
        if (r.get("reasoning") or "").strip():
            parts.append(r["reasoning"].strip())
        if r.get("final_answer") is not None:
            unit = (r.get("unit") or "").strip()
            parts.append(f"**Final answer:** {r['final_answer']}{(' ' + unit) if unit else ''}".rstrip())
        if (r.get("solver_code") or "").strip():
            parts.append(f"```python\n{r['solver_code'].strip()}\n```")
        return "\n\n".join(parts)
    return (r.get("output") or r.get("solver_code") or "").strip()   # "qa" and anything else

def to_messages(r):
    """Build a 3-part chat conversation: system -> user (question) -> assistant (answer)."""
    user = (r.get("instruction") or "").strip()
    if (r.get("input") or "").strip():
        user += "\n\nInput:\n" + r["input"].strip()
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": build_answer(r)},
    ]

print("Helpers ready.")


## Steps 1–2 — Load, clean, and remove duplicates

**What:** read every `.json`/`.jsonl` file you attached under `/kaggle/input/`, drop rows with no question or no answer, then keep only **one row per unique question**.

**Why:**
- Empty rows teach nothing.
- Duplicate questions cause **memorization** and, worse, can **leak** across the train/validation split later.

**What could go wrong:**
- *Nothing loads* → you forgot **+ Add Input**, or the file has a different name. The code prints exactly what it found so you can see.
- *Only ~88 unique* → you attached only the original 638-row file. Attach the **combined** file (your data + EE + physics) for the full ~3,200 unique — or regenerate it with `scripts/01_fetch_external.py`.
- *Too aggressive cleaning* → we only merge questions that are **identical after normalizing** (lowercase, punctuation/spaces removed), which is safe.

**How we check:** the printout shows total rows, a breakdown by source, how many empty rows were dropped, and the final unique-question count.


In [ ]:
# ---------- Step 1: LOAD every dataset file you attached under /kaggle/input ----------
INPUT_DIR = "/kaggle/input"
paths = sorted(glob.glob(f"{INPUT_DIR}/**/*.jsonl", recursive=True) +
               glob.glob(f"{INPUT_DIR}/**/*.json",  recursive=True))
print("Files found on Kaggle:")
for p in paths:
    print("  ", p)

data = []
for p in paths:
    data += read_jsonl(p)

# Fallback so this also runs outside Kaggle (e.g. inside your local repo):
if not data:
    for alt in ["data/external/all_combined.jsonl", "data/raw/data.json"]:
        if os.path.exists(alt):
            data += read_jsonl(alt)

assert data, "No data found. On Kaggle, click '+ Add Input' and attach your dataset."
print(f"\nLoaded {len(data)} rows")
print("  by source:", dict(Counter(r.get('source', '?') for r in data)))
print("  by task  :", dict(Counter(r.get('task_type', '?') for r in data)))

# ---------- Step 2a: CLEAN — drop rows with no question or no answer ----------
def is_valid(r):
    return bool((r.get("instruction") or "").strip()) and bool(build_answer(r).strip())

clean = [r for r in data if is_valid(r)]
print(f"\nCleaned: kept {len(clean)} / {len(data)}  (dropped {len(data) - len(clean)} empty rows)")

# ---------- Step 2b: DEDUPE — keep ONE row per unique, normalized question ----------
seen, unique = set(), []
for r in clean:
    key = norm(r.get("instruction"))
    if key and key not in seen:
        seen.add(key)
        unique.append(r)
print(f"Deduped: {len(unique)} unique questions  (removed {len(clean) - len(unique)} repeats)")


## Steps 3–4 — Check the domain mix, then split (leakage-safe) and save

**Step 3 — domain mix.** You have three kinds of data: RF/DSP (small), electrical-engineering (medium), physics (large). If physics dominates, the model drifts toward general physics instead of your RF focus. Set `PHYSICS_CAP` if you want to keep fewer physics rows.

**Step 4 — the split.** We split the **unique** rows into **90% train / 10% validation**, keeping each source fairly represented (*stratified*). 
- *Train* = questions the model studies.
- *Validation* = held-out questions we watch during training to catch overfitting.

**Why leakage-safe:** because we already kept only one row per unique question, the same question **cannot** land in both train and val. We still **verify** it (overlap must be `0`).

**What could go wrong:** a question sneaking into both sides → the `assert` will stop the notebook if that ever happens.

**How we check:** `overlap: 0`, and two files (`train.jsonl`, `val.jsonl`) saved to `/kaggle/working/`. They're saved as chat messages (`system → user → assistant`) — exactly what the GPU training step (Day 4) expects.


In [ ]:
# ---------- Step 3: DOMAIN MIX (physics can dominate; cap it here if you want) ----------
print("Unique rows by source:", dict(Counter(r.get('source', '?') for r in unique)))

PHYSICS_CAP = None   # e.g. set to 1000 to keep at most 1000 physics rows; None = keep all
if PHYSICS_CAP is not None:
    phys   = [r for r in unique if "physics" in r.get("source", "").lower()][:PHYSICS_CAP]
    others = [r for r in unique if "physics" not in r.get("source", "").lower()]
    unique = others + phys
    random.shuffle(unique)
    print("After physics cap    :", dict(Counter(r.get('source', '?') for r in unique)))

# ---------- Step 4: LEAKAGE-SAFE 90/10 SPLIT, stratified by source ----------
buckets = defaultdict(list)
for r in unique:
    buckets[r.get("source", "?")].append(r)

train, val = [], []
VAL_FRAC = 0.10
for src, rows in buckets.items():
    random.shuffle(rows)
    k = max(1, int(len(rows) * VAL_FRAC))   # at least 1 val row per source
    val   += rows[:k]
    train += rows[k:]
random.shuffle(train)
random.shuffle(val)

# VERIFY no question is in both sides (this MUST be 0)
overlap = {norm(r["instruction"]) for r in train} & {norm(r["instruction"]) for r in val}
print(f"\ntrain: {len(train)}  |  val: {len(val)}  |  overlap: {len(overlap)} (must be 0)")
assert len(overlap) == 0, "LEAKAGE! a question appears in both train and val."

# ---------- Save as chat-formatted JSONL (what the GPU training step expects) ----------
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({"messages": to_messages(r)}, ensure_ascii=False) + "\n")

dump(f"{OUT}/train.jsonl", train)
dump(f"{OUT}/val.jsonl",   val)
print("Saved:", f"{OUT}/train.jsonl", "and", f"{OUT}/val.jsonl")

print("\nExample formatted row (first train item):")
for m in to_messages(train[0]):
    print(f"  [{m['role']}] {m['content'][:140]}")


## Step 5 — Start the 100-question benchmark (you write these by hand)

**What:** create 100 multiple-choice questions (options A–D, exactly one correct) that the model has **never seen**, spread across your topics: Modulation, DSP, Wireless, RF Fundamentals — and decide how to handle **SDR** (your data has none, so those questions test general knowledge, not learned content).

**Why:** this is the *exam*. It must be independent of the training data, or the final score is meaningless. This is how we'll fairly measure "before vs after".

**Rules for good MCQs:**
- Exactly **one** correct answer; wrong options should be *plausible*, not silly.
- **Don't copy** a training question — test the *idea*, in your own words.
- **Balance the answer letters** (don't make them all "B").
- Aim for **~20 per domain** (5 domains × 20 = 100).

The next cell has 5 worked examples, the exact JSON format, and a **validator** that checks each question. Copy the pattern and **expand it to 100**, then save as `rf_mcq_100.jsonl`.


In [ ]:
# The exam = multiple-choice questions the model has NEVER seen. You write these BY HAND.
# Below: 5 worked examples + the exact format. Copy the pattern and grow this to 100.
mcq_examples = [
    {"id": "mcq-001", "domain": "RF Fundamentals", "difficulty": "easy",
     "question": "If a signal's power doubles, by how many dB does it increase?",
     "options": {"A": "2 dB", "B": "3 dB", "C": "6 dB", "D": "10 dB"},
     "answer": "B", "rationale": "10*log10(2) ≈ 3.01 dB."},
    {"id": "mcq-002", "domain": "DSP", "difficulty": "easy",
     "question": "A 30 kHz tone is sampled at 48 kHz. What alias frequency appears?",
     "options": {"A": "6 kHz", "B": "12 kHz", "C": "18 kHz", "D": "No aliasing"},
     "answer": "C", "rationale": "|30 - 48| = 18 kHz, which is below Nyquist (24 kHz)."},
    {"id": "mcq-003", "domain": "Modulation", "difficulty": "medium",
     "question": "How many bits does each symbol carry in 64-QAM?",
     "options": {"A": "4", "B": "6", "C": "8", "D": "16"},
     "answer": "B", "rationale": "log2(64) = 6 bits per symbol."},
    {"id": "mcq-004", "domain": "Wireless", "difficulty": "medium",
     "question": "In free space, doubling the distance increases path loss by about how many dB?",
     "options": {"A": "3 dB", "B": "6 dB", "C": "12 dB", "D": "20 dB"},
     "answer": "B", "rationale": "FSPL rises as 20*log10(d); doubling d adds 20*log10(2) ≈ 6 dB."},
    {"id": "mcq-005", "domain": "RF Fundamentals", "difficulty": "easy",
     "question": "0 dBm is equal to how much power?",
     "options": {"A": "0 mW", "B": "1 mW", "C": "10 mW", "D": "100 mW"},
     "answer": "B", "rationale": "0 dBm = 1 mW by definition."},
]

# ---------- Validator: catches format mistakes before they waste your time ----------
REQUIRED = {"id", "domain", "difficulty", "question", "options", "answer", "rationale"}

def validate_mcq(q):
    problems = []
    if not REQUIRED.issubset(q):
        problems.append(f"missing keys: {REQUIRED - set(q)}")
    if set(q.get("options", {})) != {"A", "B", "C", "D"}:
        problems.append("options must be exactly A, B, C, D")
    if q.get("answer") not in {"A", "B", "C", "D"}:
        problems.append("answer must be one of A/B/C/D")
    return problems

for q in mcq_examples:
    issues = validate_mcq(q)
    print(q["id"], "OK" if not issues else issues)

print("\nDomain counts so far:", dict(Counter(q["domain"] for q in mcq_examples)),
      "  (target: ~20 each, 100 total)")
print("Answer-letter balance:", dict(Counter(q["answer"] for q in mcq_examples)),
      "  (keep this roughly even)")

with open(f"{OUT}/rf_mcq_starter.jsonl", "w", encoding="utf-8") as f:
    for q in mcq_examples:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
print("\nSaved starter:", f"{OUT}/rf_mcq_starter.jsonl", "-> expand to 100 and save as rf_mcq_100.jsonl")


## Step 6 — Leakage audit (test vs training)

**What:** for each exam question, measure its highest word-overlap with **any** training question.

**Why:** to *prove* the exam isn't secretly reusing study material — the thing that would make your final score fake.

**How we check:** all scores should be **low**. Anything **≥ 0.60** gets flagged so you can reword it. (Run this again after you expand to 100 questions.)

---

## What you produced today
- `train.jsonl` and `val.jsonl` — clean, deduped, chat-formatted, **0 leakage** (verified).
- `rf_mcq_starter.jsonl` — the start of your 100-question exam (finish it by hand).
- A verified split and a leakage audit.

### Save your work on Kaggle (important!)
Files in `/kaggle/working/` are **deleted when the session ends** unless you click **Save Version** (top-right). Do that now — you can also download them from the **Output** tab.

### Day 2 checklist
- [ ] Data loaded and inspected
- [ ] Cleaned (no empty rows)
- [ ] Deduped to unique questions
- [ ] Domain mix reviewed
- [ ] 90/10 split with **overlap = 0**
- [ ] `train.jsonl` / `val.jsonl` saved
- [ ] 100-question benchmark started (finish by hand → `rf_mcq_100.jsonl`)
- [ ] Leakage audit run

### Next — Day 3
Build the grading harness and measure the **base model** (untouched Qwen2.5-7B) on your 100 questions. That's your "before" score. **Day 4** does the actual QLoRA training on a GPU (this is where you switch the Kaggle accelerator **on**).


In [ ]:
# Compare each exam question to ALL training questions. High overlap = possible leak.
train_sets = [word_set(r["instruction"]) for r in train]

def max_overlap(text):
    s = word_set(text)
    return max((jaccard(s, t) for t in train_sets), default=0.0)

print("Leakage check — MCQ vs training questions (lower is better):")
worst = 0.0
for q in mcq_examples:
    score = max_overlap(q["question"])
    worst = max(worst, score)
    flag = "   <-- REVIEW: too similar to training!" if score >= 0.60 else ""
    print(f"  {q['id']}  max-similarity = {score:.2f}{flag}")

print(f"\nHighest similarity found: {worst:.2f}  (rule of thumb: keep every question < 0.60)")
print("Re-run this cell after you expand the exam to 100 questions.")
